# 04 · Clasificación binaria: alerta de demanda alta


**Ruta sugerida:** lee la teoría, ejecuta las celdas en orden, predice lo que ocurrirá, realiza el ejercicio y luego compara con la solución. Puedes modificar y volver a ejecutar las celdas. Los datos son **sintéticos**: ilustran un contexto, no representan observaciones reales.

**Conceptos compartidos.** Una neurona calcula una combinación de entradas y un sesgo; una activación puede transformar el resultado. Durante `fit`, el optimizador ajusta los pesos para reducir una pérdida mediante gradientes. Una época es una pasada por los datos de entrenamiento. Validación ayuda a decidir cambios; prueba se reserva para la evaluación final. El descenso de entrenamiento con validación creciente puede indicar sobreajuste; ambas pérdidas altas pueden indicar subajuste. Estas son señales, no diagnósticos automáticos.

**Preparación:** Google Colab ya suele incluir las dependencias. En Jupyter local instala en el entorno activo `pip install tensorflow numpy matplotlib scikit-learn`; reinicia el kernel si acabas de instalar. Ejecuta todas las celdas en orden. El entrenamiento usa CPU y puede tardar unos minutos según el equipo.


**Tiempo orientativo:** 25–35 minutos.

## 1. Idea central

La salida sigmoide está entre 0 y 1 y puede interpretarse como una probabilidad estimada. La entropía cruzada binaria penaliza predicciones seguras y equivocadas; la exactitud depende del umbral y puede ocultar errores por clase. Estimaremos una alerta sintética a partir de temperatura y horas de operación.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

keras.utils.set_random_seed(42)
np.random.seed(42)
print("TensorFlow:", tf.__version__)

## 2. Datos y separación

Primero observa las formas: cada fila es un ejemplo y cada columna una característica. Reservamos 60 % para entrenamiento, 20 % para validación y 20 % para prueba.

In [ ]:
rng = np.random.default_rng(42)
temperatura = rng.uniform(10, 35, 1000)
horas = rng.uniform(1, 12, 1000)
X = np.column_stack([temperatura, horas]).astype("float32")
logit = 0.19 * (temperatura - 22) + 0.55 * (horas - 6.5)
prob_real = 1 / (1 + np.exp(-logit))
y = rng.binomial(1, prob_real).astype("float32")

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp
)
escalador = StandardScaler().fit(X_train)
X_train_s = escalador.transform(X_train).astype("float32")
X_val_s = escalador.transform(X_val).astype("float32")
X_test_s = escalador.transform(X_test).astype("float32")
print(
    "Formas entrenamiento / validación / prueba:",
    X_train.shape,
    X_val.shape,
    X_test.shape,
)

## 3. Modelo y entrenamiento

Antes de ejecutar: ¿qué esperas que ocurra con la pérdida de entrenamiento y de validación?

In [ ]:
modelo = keras.Sequential(
    [
        keras.Input(shape=(2,)),
        keras.layers.Dense(8, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ]
)
modelo.compile(
    optimizer=keras.optimizers.Adam(0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
modelo.summary()
historia = modelo.fit(
    X_train_s,
    y_train,
    validation_data=(X_val_s, y_val),
    epochs=60,
    batch_size=32,
    verbose=0,
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(historia.history["loss"], label="Entrenamiento")
plt.plot(historia.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Conversación:** ¿bajan ambas curvas? ¿Cuándo se estabilizan? ¿Hay separación entre ellas? Recuerda que la pérdida de validación puede fluctuar por el tamaño de la muestra.

## 4. Primera evaluación

Esta es una evaluación ilustrativa del modelo inicial. Si lo modificas a partir de estos resultados, el conjunto de prueba deja de ser una evaluación final independiente. Usa validación para elegir y conserva otra prueba final si ya la consultaste.

In [ ]:
resultado = dict(
    zip(modelo.metrics_names, modelo.evaluate(X_test_s, y_test, verbose=0))
)
print("Evaluación de prueba:", resultado)
pred = modelo.predict(X_test_s, verbose=0).ravel()

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

etiquetas = (pred >= 0.5).astype(int)
ConfusionMatrixDisplay(
    confusion_matrix(y_test, etiquetas), display_labels=["Sin alerta", "Alerta"]
).plot(cmap="Blues")
plt.show()
print(
    classification_report(
        y_test, etiquetas, target_names=["Sin alerta", "Alerta"], zero_division=0
    )
)

## 5. Tu turno

Usa validación para comparar umbrales 0,3 y 0,7. Calcula precisión y sensibilidad de la clase alerta. ¿Cuál elegirías si no detectar una alerta fuera más costoso? Justifica el compromiso.

Escribe tu código en la siguiente celda y responde en la siguiente celda de texto. Luego consulta la solución.

In [ ]:
# Tu código aquí

**Mi interpretación:** escribe aquí tu respuesta.

## 6. Solución de referencia

Intenta resolverlo antes de ejecutar la celda. Los resultados pueden variar ligeramente entre versiones y dispositivos.

In [ ]:
from sklearn.metrics import precision_score, recall_score

p_val = modelo.predict(X_val_s, verbose=0).ravel()
for umbral in [0.3, 0.7]:
    clases = (p_val >= umbral).astype(int)
    print(
        f"Umbral {umbral}: precisión={precision_score(y_val, clases, zero_division=0):.3f}; sensibilidad={recall_score(y_val, clases, zero_division=0):.3f}"
    )

Bajar el umbral suele aumentar la sensibilidad y también las falsas alarmas; la precisión puede disminuir. Si omitir alertas es más costoso, 0,3 puede ser razonable, pero la decisión requiere costes y validación. Tras elegir el umbral, evalúalo una sola vez en prueba.

## 7. Desafío opcional

Traza precisión y sensibilidad de validación para umbrales entre 0,1 y 0,9; elige uno y evalúalo en prueba.

**Cierre:** menciona una conclusión de las curvas y una limitación de estos datos sintéticos.